# 05 · MC dropout (OPTIONAL, run last, Colab T4)

**Optional.** Run it only if notebooks 01–04 went cleanly. The project is complete without it.

Takes the seed 42 ResNet-50 from notebook 01, keeps its backbone exactly as trained, puts a
dropout layer (p = 0.5) before the final layer and retrains that head (5 epochs). Then 20
stochastic passes per image: the mean softmax is the prediction, the spread the uncertainty
(predictive variance, and mutual information as the fourth score). Same sweep as the single
model and the ensemble, and one figure comparing all three methods (DECISIONS.md D18).

**Needs:** notebooks 01 and 02 (the seed 42 run and its sweep). The ensemble from 03 is
optional; without it the figure compares two methods and says so. **After a disconnect:**
run all again; a finished head or sweep in Drive is reused.

In [ ]:
# ---- settings: the only cell you normally edit ---------------------------------
REPO_URL = "https://github.com/brycedaniliuc/COMP2035-grape-calibration.git"
REPO_REF = "main"                       # a branch, tag or commit hash
ZIP_PATH = "/content/drive/MyDrive/grape_dataset_for_colab.zip"   # must match Drive
DATA_ROOT = "/content/grape_dataset"    # the zip extracts to this folder
OUT = "/content/drive/MyDrive/COMP2035_runs"   # every output is written here, as it goes

In [ ]:
import os, subprocess, sys, time
from pathlib import Path
T0 = time.time()

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")

REPO = Path("/content/COMP2035-grape-calibration")
if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "--tags"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True,
                        capture_output=True, text=True).stdout.strip()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("code at", REPO_REF, commit)

# Colab already has torch, torchvision, numpy, scikit-learn, scikit-image and matplotlib.
# Do NOT pip install requirements.txt here: it pins the CPU build of torch.
import importlib.util
missing = [m for m in ("skimage", "sklearn", "nbformat") if importlib.util.find_spec(m) is None]
if missing:
    pkg = {"skimage": "scikit-image", "sklearn": "scikit-learn"}
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *[pkg.get(m, m) for m in missing]], check=True)

from src.train import environment, select_device
import json
env = environment(select_device())
print(json.dumps(env, indent=1))
if env["device"] != "cuda":
    print("!! NO GPU: Runtime > Change runtime type > T4 GPU, then run all again")
os.makedirs(OUT, exist_ok=True)
print("outputs ->", OUT)

In [ ]:
# Fail early: unzip and check the exact per-folder counts BEFORE any training.
import shutil, zipfile
if not os.path.isdir(DATA_ROOT):
    assert os.path.exists(ZIP_PATH), (
        f"{ZIP_PATH} not found. Put grape_dataset_for_colab.zip at the top level of My Drive. "
        f"Files there: {sorted(os.listdir('/content/drive/MyDrive'))[:40]}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall("/content/")
from src.data import check_supplied_dataset
check_supplied_dataset(DATA_ROOT)       # raises unless 7,222 train + 1,805 test, per class
print(f"elapsed {(time.time() - T0) / 60:.1f} min")

## The seed 42 run (notebook 01) and its sweep (notebook 02)

In [ ]:
from src.ensemble import find_run
BASE = find_run(OUT, seed=42)
assert BASE, f"no finished resnet50 source run in {OUT}: run notebook 01 first"
RUN_ID = json.loads(BASE.read_text())["run_id"]
assert (BASE.parent / f"sweep_{RUN_ID}.json").exists(), "no sweep: run notebook 02 first"
print("model:", BASE.name)

## Retrain the head with dropout, then 20 passes in every condition (about 10 min)

In [ ]:
from src.evaluate import print_threshold
from src.mcdropout import mc_dropout
MC = mc_dropout(BASE, data_root=DATA_ROOT)
MC_PATH = f"{OUT}/mcdropout_{MC['mc_id']}.json"
print_threshold(MC["threshold"])
print(f"elapsed {(time.time() - T0) / 60:.1f} min")

## Single model vs ensemble vs MC dropout at every severity, to Drive

In [ ]:
from src.figures import render_methods
paths = render_methods(MC_PATH, f"{OUT}/figures_{RUN_ID}", tag=RUN_ID)
print(len(paths), "files ->", f"{OUT}/figures_{RUN_ID}")

## Verify (Tier 1 must pass)

In [ ]:
r = subprocess.run([sys.executable, "-m", "src.verify", "--mcdropout", MC_PATH],
                   capture_output=True, text=True)
print(r.stdout[-6000:], r.stderr[-3000:])
assert r.returncode == 0, "a Tier 1 check failed: stop and investigate"
print(f"total elapsed {(time.time() - T0) / 60:.1f} min")

## Everything this notebook wrote to Drive

In [ ]:
for p in sorted(Path(OUT).rglob("*")):
    if p.is_file() and ("mcdropout" in p.name or "methods_comparison" in p.name):
        print(f"{p.stat().st_size / 1e6:9.2f} MB  {p.relative_to(OUT)}")